# ANLP — Session 1: NLP foundations & text pre-processing

**Hands-on notebook** · blocks: *Env setup* · *Tokenization* · *Exploratory first look (Zipf)* · *Pre-processing payoff*

Work through the sections top to bottom. Cells marked **👉 Your turn** are exercises.

## 0. Environment setup

If running this notebook locally, ensure you have a Python 3.12 environment set up. Follow the instructions in `README.md` to set up the environment and install the required packages. If you are using **Google Colab**, the next cell will install everything you need automatically.

In [ ]:
%%capture
import sys

# Install dependencies when running in Google Colab.
# On your own computer, set up the environment as described in README.md instead.
if "google.colab" in sys.modules:
    print("Google Colab environment detected. Installing dependencies...")

    # install
    # (one-time; takes a few minutes incl. model downloads; covers all four sessions)
    # (torch is pre-installed on Colab, so it is not pinned here)
    %pip install spacy==3.8.14 pandas==3.0.3 numpy==2.5.4 matplotlib==3.11.0 wordcloud==1.9.6 scikit-learn==1.9.0 gensim==4.4.0 sentence-transformers==5.6.0 transformers==5.12.1 keybert==0.9.0 langdetect==1.0.9 datasets==5.0.0 nltk==3.9.4
    # download the small English spaCy model (language data, shipped separately from the library)
    !python -m spacy download en_core_web_sm-3.8.0

In [ ]:
# Sanity check: imports and spaCy model
import spacy, pandas, matplotlib, wordcloud, sklearn
# nlp is the loaded spaCy pipeline — we reuse this one object everywhere below
nlp = spacy.load("en_core_web_sm")
print("spaCy", spacy.__version__, "| model loaded:", nlp.meta["name"], "✓")

In [ ]:
# Load our working corpus: "Alice's Adventures in Wonderland" (public domain, Project Gutenberg)
# Falls back to a small built-in sample if you are offline.
import urllib.request

FALLBACK = (
    "Alice was beginning to get very tired of sitting by her sister on the bank, "
    "and of having nothing to do: once or twice she had peeped into the book her sister "
    "was reading, but it had no pictures or conversations in it, 'and what is the use of "
    "a book,' thought Alice 'without pictures or conversations?' "
    "So she was considering in her own mind (as well as she could, for the hot day made her "
    "feel very sleepy and stupid), whether the pleasure of making a daisy-chain would be worth "
    "the trouble of getting up and picking the daisies, when suddenly a White Rabbit with pink "
    "eyes ran close by her. There was nothing so very remarkable in that; nor did Alice think it "
    "so very much out of the way to hear the Rabbit say to itself, 'Oh dear! Oh dear! I shall be late!'"
)

try:
    url = "https://www.gutenberg.org/files/11/11-0.txt"
    # download the page and decode the raw bytes into a Python string
    raw = urllib.request.urlopen(url, timeout=10).read().decode("utf-8")
    # strip the Gutenberg header/footer. .find() returns the character index where a marker begins
    start = raw.find("*** START OF")
    start = raw.find("\n", start) + 1   # jump to the start of the line AFTER the marker
    end = raw.find("*** END OF")
    text = raw[start:end].strip()       # keep only the text between the two markers
    print(f"Downloaded corpus: {len(text):,} characters")
except Exception as e:
    # any network problem lands here, so the lesson still runs offline
    text = FALLBACK
    print(f"Download failed ({e}); using built-in sample: {len(text):,} characters")

## 1 · Tokenization

Splitting text into units (tokens) sounds trivial — until you meet contractions, abbreviations, URLs, and punctuation. Let's see why `str.split()` is not enough.

In [ ]:
sample = "Dr. Smith didn't visit Zurich in 2024. She loved it! Visit https://uzh.ch."

# Naive whitespace tokenization: str.split() breaks on spaces only,
# so punctuation stays glued to words ("it!", "2024.", the trailing ".")
naive_tokens = sample.split()
print("naive:", naive_tokens)

In [ ]:
# spaCy tokenization
doc = nlp(sample)
# loop over the Doc: each t is a Token object; t.text is its string form
spacy_tokens = [t.text for t in doc]
print("spaCy:", spacy_tokens)

Compare the two outputs. Where do they differ? Look at `didn't`, the sentence-final punctuation, and the URL.

### How spaCy represents text

`nlp("...")` doesn't return a list of strings — it returns a **`Doc`**, spaCy's container
for analysed text. Three object types matter:

- **`Doc`** — the whole text, as a sequence of tokens.
- **`Token`** — one unit (word, punctuation, URL…). *Not a string* — a rich object with attributes.
- **`Span`** — a slice of a `Doc` (a sentence, an entity). `doc[2:5]`, and every item in `doc.sents`, is a `Span`.

Token attributes often come in **pairs** — note the trailing underscore:

| string (readable)            | hash (integer ID) | meaning              |
|------------------------------|-------------------|----------------------|
| `token.text` / `token.orth_` | `token.orth`      | the surface form     |
| `token.lemma_`               | `token.lemma`     | dictionary base form |
| `token.pos_`                 | `token.pos`       | part of speech       |

The trailing `_` gives the **human-readable string**; without it you get an **integer hash**.
spaCy stores every string once in a central table (the `Vocab`'s `StringStore`) and refers to it
by that hash everywhere — that's why it's fast and memory-light, and why you'll almost always want
the `_` version. You can look a hash up in both directions via `doc.vocab.strings`.

(The boolean flags `token.is_stop`, `token.is_punct`, `token.is_alpha` are exactly what we use to clean the text in §3.)

In [ ]:
# A peek at spaCy's object model
demo = nlp("The Rabbit checked its watch.")

print(type(demo).__name__, "->", type(demo[0]).__name__, "/", type(demo[1:3]).__name__)  # Doc -> Token / Span

# Token attributes: string form (with _) vs the integer hash (without)
print(f"\n{'text':>8} {'lemma_':>8} {'pos_':>6}  | is_stop")
for t in demo:
    print(f"{t.text:>8} {t.lemma_:>8} {t.pos_:>6}  | {t.is_stop}")

# Strings are stored once and referenced by hash — the StringStore, in both directions
h = demo.vocab.strings["watch"]
print("\n'watch' -> hash:", h, "-> back to string:", demo.vocab.strings[h])
print("checked.lemma (hash):", demo[2].lemma, " checked.lemma_ (string):", demo[2].lemma_)

In [ ]:
# Sentence tokenization — doc.sents yields one Span per sentence
# enumerate(..., 1) pairs each sentence with a counter that starts at 1
for i, sent in enumerate(doc.sents, 1):
    print(i, sent.text)

**👉 Your turn** — run the corpus through spaCy. How many tokens and sentences does it contain? Find three tokens where whitespace splitting would have gone wrong.

In [ ]:
# text[:50_000] = the first 50,000 characters (the underscores are just digit separators)
corpus_doc = nlp(text[:50_000])  # cap length to keep it snappy in class

n_tokens = len(corpus_doc)              # len() of a Doc = number of tokens
n_sents = len(list(corpus_doc.sents))   # .sents is a generator, so list() it before counting
print(f"{n_tokens:,} tokens · {n_sents:,} sentences")

# your exploration here ...

## 2 · Exploratory first look: Zipf's law

Before cleaning anything, let's look at the **raw** frequency distribution. Keep these plots — we re-run them after pre-processing.

In [ ]:
from collections import Counter
import matplotlib.pyplot as plt

raw_tokens = [t.text for t in corpus_doc]
# Counter tallies how many times each token string appears -> {token: count}
raw_freqs = Counter(raw_tokens)
# .most_common(10) = the 10 (token, count) pairs with the highest counts
print("Most common (raw):", raw_freqs.most_common(10))

In [ ]:
# Zipf plot: rank vs. frequency on log-log axes
# sort the counts high -> low; an item's position in this list is its rank (1 = most frequent)
counts = sorted(raw_freqs.values(), reverse=True)
plt.figure(figsize=(6, 4))
# loglog puts BOTH axes on a log scale, which makes Zipf's law look roughly like a straight line.
# x = ranks 1..N, y = the sorted counts; markers only, no connecting line
plt.loglog(range(1, len(counts) + 1), counts, marker=".", linestyle="none")
plt.xlabel("rank"); plt.ylabel("frequency"); plt.title("Zipf plot — RAW tokens")
plt.show()

A few words dominate; a very long tail of words that appear only once or twice. **What are the dominant words — do they carry the story's content?**

In [ ]:
# Optional stretch: word cloud (illustration only — size ≈ frequency, nothing more!)
from wordcloud import WordCloud
# Filter out tokens that contain newlines or are whitespace-only — PIL can't render them
wc_freqs = {k: v for k, v in raw_freqs.items() if k.strip() and "\n" not in k}
# generate_from_frequencies takes our {word: count} Counter and sizes each word by its count
wc = WordCloud(width=800, height=400, background_color="white").generate_from_frequencies(wc_freqs)
plt.figure(figsize=(8, 4)); plt.imshow(wc, interpolation="bilinear"); plt.axis("off")
plt.title("Word cloud — RAW tokens"); plt.show()

In [ ]:
# Baseline measurements — we track these through every cleaning step
def vocab_stats(tokens, label):
    """Print token count, type count, and TTR for a token list.

    Args:
        tokens: list of token strings to analyse.
        label:  short name for this processing stage (e.g. "raw", "lemmatized"),
                printed left-aligned so successive calls form a readable table.
    """

    # set(tokens) keeps each distinct token once, so its length = number of *types* (unique words)
    # TTR = types / tokens; a lower value means more repetition
    ttr = len(set(tokens)) / len(tokens)
    # format specs: >28 right-aligns in 28 chars, "," adds thousands separators, .4f = 4 decimals
    print(f"{label:>28}: {len(tokens):>7,} tokens · {len(set(tokens)):>6,} types · TTR {ttr:.4f}")

vocab_stats(raw_tokens, "raw")

## 3 · Pre-processing: the payoff

Now we clean step by step — and **measure** what each step does to the vocabulary.

In [ ]:
# Step 1: lowercase — folds "The", "the", and "THE" into one form
lower_tokens = [t.text.lower() for t in corpus_doc]
vocab_stats(raw_tokens, "raw")
vocab_stats(lower_tokens, "lowercased")

In [ ]:
# Step 2: drop punctuation, whitespace-only tokens, and stopwords.
# Read the comprehension as: keep t.text.lower() for each token t, UNLESS it is
# punctuation, whitespace, or a stopword (the is_* flags are booleans on every Token).
content_tokens = [
    t.text.lower() for t in corpus_doc
    if not (t.is_punct or t.is_space or t.is_stop)
]
vocab_stats(content_tokens, "− punct/stopwords")

In [ ]:
# Step 3: lemmatize — spaCy's dictionary-based base forms (e.g. running -> run).
# Same filter as before, but we collect t.lemma_ instead of the surface text.
lemmas = [
    t.lemma_.lower() for t in corpus_doc
    if not (t.is_punct or t.is_space or t.is_stop)
]
vocab_stats(lemmas, "lemmatized")

The vocabulary shrinks substantially while the text still carries its meaning — that's the point of the "tedious" steps.

In [ ]:
# Re-run the Zipf plot on CLEANED tokens — compare with the raw plot above
clean_freqs = Counter(lemmas)
print("Most common (cleaned):", clean_freqs.most_common(10))

# same plotting recipe as the raw Zipf plot, now on the cleaned counts
counts = sorted(clean_freqs.values(), reverse=True)
plt.figure(figsize=(6, 4))
plt.loglog(range(1, len(counts) + 1), counts, marker=".", linestyle="none")
plt.xlabel("rank"); plt.ylabel("frequency"); plt.title("Zipf plot — CLEANED tokens")
plt.show()

In [ ]:
# And the cleaned word cloud — now the content words surface
wc = WordCloud(width=800, height=400, background_color="white").generate_from_frequencies(clean_freqs)
plt.figure(figsize=(8, 4)); plt.imshow(wc, interpolation="bilinear"); plt.axis("off")
plt.title("Word cloud — CLEANED tokens"); plt.show()

**👉 Your turn** — compare the raw and cleaned most-common lists. Which words moved up? What fraction of all tokens were stopwords?

In [ ]:
# your analysis here ...

### Optional: stemming vs. lemmatization

spaCy deliberately ships **no stemmer** (lemmatization only). To see what crude suffix-chopping does, this one cell uses NLTK's SnowballStemmer — the single exception to our spaCy-only rule.

In [ ]:
from nltk.stem.snowball import SnowballStemmer
stemmer = SnowballStemmer("english")

examples = ["studies", "studying", "ran", "running", "was", "universities", "better"]
print(f"{'token':>14} {'stem':>12} {'lemma':>12}")
for w in examples:
    # nlp(w)[0] = run spaCy on the single word and take its first (and only) Token
    lemma = nlp(w)[0].lemma_
    # stemmer.stem() chops suffixes (crude); lemma_ looks up a real base form (dictionary)
    print(f"{w:>14} {stemmer.stem(w):>12} {lemma:>12}")

## 4 · Beyond single tokens: n-grams

Every count so far has treated the text as a **bag of single tokens** (unigrams), throwing
word order away. An **n-gram** is a contiguous run of *n* tokens:

- **unigram** (n = 1): `alice`
- **bigram**  (n = 2): `white rabbit`
- **trigram** (n = 3): `mock turtle story`

n-grams capture local context that single tokens miss — multi-word names (`white rabbit`),
fixed phrases, and crucially **negation**: `not good` survives as a bigram even though the
unigrams `not` and `good` look unrelated on their own. (Recall the pitfall from the slides:
dropping the stopword `not` flips sentiment — keeping bigrams is one way to preserve it.)

In [ ]:
# An n-gram is just a sliding window of length n over the token list.
def make_ngrams(tokens, n):
    # at each start position i, take the slice tokens[i : i+n] and store it as a tuple
    return [tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1)]

demo = ["the", "movie", "was", "not", "good"]
print("unigrams:", make_ngrams(demo, 1))
print("bigrams :", make_ngrams(demo, 2))   # ('not', 'good') stays together
print("trigrams:", make_ngrams(demo, 3))

In [ ]:
# Most common bigrams / trigrams on our CLEANED tokens (lemmas — stopwords already removed):
# these surface content collocations.
print("Top bigrams (cleaned): ", Counter(make_ngrams(lemmas, 2)).most_common(8))
print("Top trigrams (cleaned):", Counter(make_ngrams(lemmas, 3)).most_common(5))

# Contrast: keep stopwords (drop only punctuation/space) and the top bigrams flip to
# function-word pairs — Zipf's law again, now over pairs of words.
kept = [t.text.lower() for t in corpus_doc if not (t.is_punct or t.is_space)]
print("\nWith stopwords kept:    ", Counter(make_ngrams(kept, 2)).most_common(8))

### N-gram sparsity: Zipf explodes as *n* grows

Each extra token in the window multiplies the **type** (unique tokens) count while spreading the same number of tokens thinner. Plot all three orders on the same log-log axes and the tail grows visibly — and the fraction of features seen only **once** (hapax legomena) shoots up fast. That's the curse of n-gram features: most of your columns will hold a single count.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

hapax_rows = []
for n, label, color in [(1, "unigrams", "steelblue"),
                         (2, "bigrams",  "darkorange"),
                         (3, "trigrams", "seagreen")]:
    freqs = Counter(make_ngrams(lemmas, n))
    counts_n = sorted(freqs.values(), reverse=True)
    hapax = sum(1 for c in counts_n if c == 1)
    hapax_pct = 100 * hapax / len(counts_n)
    hapax_rows.append((n, len(counts_n), hapax, hapax_pct))

    # connected line makes the tail's rightward extent readable; alpha keeps curves distinct
    ax.loglog(range(1, len(counts_n) + 1), counts_n,
              color=color, linewidth=1.2, alpha=0.8,
              label=f"{label}  ({len(counts_n):,} types, {hapax_pct:.0f}% hapax)")

# hapax floor: every type to the right of where a curve hits this line appears only once
ax.axhline(1, color="grey", linewidth=0.8, linestyle="--", label="hapax floor (freq = 1)")
ax.set_xlabel("rank (log)")
ax.set_ylabel("frequency (log)")
ax.set_title("Zipf plot — unigrams vs. bigrams vs. trigrams (cleaned)")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

print("Feature-space growth and sparsity:")
print(f"{'n':>2}  {'types':>8}  {'hapax legomena':>16}  {'% hapax':>8}")
for n, types, hapax, hapax_pct in hapax_rows:
    print(f"{n:>2}  {types:>8,}  {hapax:>16,}  {hapax_pct:>7.1f}%")

### Caveat to take home

Cleaning and pre-processing reshapes the signal **for classical frequency-based methods**. It is *task-dependent*, not a universal pre-processing step.